In [1]:
import json
import pandas as pd

import re
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /home/irene/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
with open("../data/raw_jobs.json", "r", encoding="utf-8") as f:
    raw_jobs = json.load(f)

df = pd.DataFrame(raw_jobs)

In [3]:
print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")

Shape: (897, 37)

Columns:
['relevance', 'id', 'external_id', 'original_id', 'label', 'webpage_url', 'logo_url', 'headline', 'application_deadline', 'number_of_vacancies', 'description', 'employment_type', 'salary_type', 'salary_description', 'duration', 'working_hours_type', 'scope_of_work', 'access', 'employer', 'application_details', 'experience_required', 'access_to_own_car', 'driving_license_required', 'driving_license', 'occupation', 'occupation_group', 'occupation_field', 'workplace_address', 'must_have', 'nice_to_have', 'application_contacts', 'publication_date', 'last_publication_date', 'removed', 'removed_date', 'source_type', 'timestamp']


In [4]:
df_clean = df[[
    "id",
    "headline",
    "occupation",
    "occupation_group",
    "occupation_field",
    "employer",
    "workplace_address",
    "description",
    "must_have",
    "nice_to_have",
    "publication_date",
    "experience_required"
]].copy()

print(f"Reduced from {df.shape[1]} to {df_clean.shape[1]} columns")
df_clean.head(3)

Reduced from 37 to 12 columns


,id,headline,occupation,occupation_group,occupation_field,employer,workplace_address,description,must_have,nice_to_have,publication_date,experience_required
0,30745178,Data Scientist,"{'concept_id': 'ziUj_67V_Yk4', 'label': 'Data ...","{'concept_id': 'UXKZ_3zZ_ipB', 'label': 'Syste...","{'concept_id': 'apaJ_2ja_LuF', 'label': 'Data/...","{'phone_number': None, 'email': None, 'url': N...","{'municipality': 'Malmö', 'municipality_code':...",{'text': 'Vi är ett startupföretag som arbetar...,"{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [{'weight': 5, 'co...",2026-03-13T15:30:20,True
1,30455674,Data Scientist,"{'concept_id': 'ziUj_67V_Yk4', 'label': 'Data ...","{'concept_id': 'UXKZ_3zZ_ipB', 'label': 'Syste...","{'concept_id': 'apaJ_2ja_LuF', 'label': 'Data/...","{'phone_number': None, 'email': None, 'url': '...","{'municipality': 'Stockholm', 'municipality_co...",{'text': 'Northmill Bank is a challenger bank ...,"{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...",2026-01-12T09:23:40,True
2,30784437,Senior Data Scientist,"{'concept_id': 'ziUj_67V_Yk4', 'label': 'Data ...","{'concept_id': 'UXKZ_3zZ_ipB', 'label': 'Syste...","{'concept_id': 'apaJ_2ja_LuF', 'label': 'Data/...","{'phone_number': None, 'email': None, 'url': N...","{'municipality': 'Södertälje', 'municipality_c...",{'text': 'Senior Data Scientist – TRATON Finan...,"{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...",2026-03-23T11:28:15,True


In [5]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 897 entries, 0 to 896
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   897 non-null    str   
 1   headline             897 non-null    str   
 2   occupation           897 non-null    object
 3   occupation_group     897 non-null    object
 4   occupation_field     897 non-null    object
 5   employer             897 non-null    object
 6   workplace_address    897 non-null    object
 7   description          897 non-null    object
 8   must_have            897 non-null    object
 9   nice_to_have         897 non-null    object
 10  publication_date     897 non-null    str   
 11  experience_required  897 non-null    bool  
dtypes: bool(1), object(8), str(3)
memory usage: 78.1+ KB


In [6]:
# Flatten nested columns to directly access the content
df_clean["occupation_label"] = df_clean["occupation"].apply(lambda x: x.get("label") if isinstance(x, dict) else None)
df_clean["occupation_group_label"] = df_clean["occupation_group"].apply(lambda x: x.get("label") if isinstance(x, dict) else None)
df_clean["occupation_field_label"] = df_clean["occupation_field"].apply(lambda x: x.get("label") if isinstance(x, dict) else None)
df_clean["employer_name"] = df_clean["employer"].apply(lambda x: x.get("name") if isinstance(x, dict) else None)
df_clean["municipality"] = df_clean["workplace_address"].apply(lambda x: x.get("municipality") if isinstance(x, dict) else None)
df_clean["region"] = df_clean["workplace_address"].apply(lambda x: x.get("region") if isinstance(x, dict) else None)
df_clean["description_text"] = df_clean["description"].apply(lambda x: x.get("text") if isinstance(x, dict) else None)

# Drop the original nested columns
df_clean = df_clean.drop(columns=["occupation", "occupation_group", "occupation_field", 
                                   "employer", "workplace_address", "description"])

print(df_clean.shape)
df_clean.head(3)

(897, 13)


,id,headline,must_have,nice_to_have,publication_date,experience_required,occupation_label,occupation_group_label,occupation_field_label,employer_name,municipality,region,description_text
0,30745178,Data Scientist,"{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [{'weight': 5, 'co...",2026-03-13T15:30:20,True,Data scientist,Systemanalytiker och IT-arkitekter m.fl.,Data/IT,Fedge AB,Malmö,Skåne län,Vi är ett startupföretag som arbetar med datal...
1,30455674,Data Scientist,"{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...",2026-01-12T09:23:40,True,Data scientist,Systemanalytiker och IT-arkitekter m.fl.,Data/IT,Northmill Bank AB,Stockholm,Stockholms län,Northmill Bank is a challenger bank at the int...
2,30784437,Senior Data Scientist,"{'skills': [], 'languages': [], 'work_experien...","{'skills': [], 'languages': [], 'work_experien...",2026-03-23T11:28:15,True,Data scientist,Systemanalytiker och IT-arkitekter m.fl.,Data/IT,Scania CV AB,Södertälje,Stockholms län,Senior Data Scientist – TRATON Financial Servi...


In [7]:
# There are a must_have and a nice_to_have sections, I want to check how many jobs have those sections populated with structured skills
has_must_have = df_clean["must_have"].apply(
    lambda x: len(x.get("skills", [])) > 0 if isinstance(x, dict) else False
).sum()

has_nice_to_have = df_clean["nice_to_have"].apply(
    lambda x: len(x.get("skills", [])) > 0 if isinstance(x, dict) else False
).sum()

print(f"Jobs with must_have skills: {has_must_have} / {len(df_clean)}")
print(f"Jobs with nice_to_have skills: {has_nice_to_have} / {len(df_clean)}")

Jobs with must_have skills: 12 / 897
Jobs with nice_to_have skills: 15 / 897


will collect the skills for a later stage

In [11]:
df_clean = df_clean.drop(columns=["must_have", "nice_to_have"])

print(df_clean.shape)
print(df_clean.columns.tolist())

(897, 11)
['id', 'headline', 'publication_date', 'experience_required', 'occupation_label', 'occupation_group_label', 'occupation_field_label', 'employer_name', 'municipality', 'region', 'description_text']


In [12]:
# Converting the date in the proprer format
df_clean["publication_date"] = pd.to_datetime(df_clean["publication_date"]).dt.date

print(df_clean["publication_date"].head())

0    2026-03-13
1    2026-01-12
2    2026-03-23
3    2026-03-20
4    2026-02-05
Name: publication_date, dtype: object


In [13]:
df_clean.isnull().sum()

id                         0
headline                   0
publication_date           0
experience_required        0
occupation_label           0
occupation_group_label     0
occupation_field_label     0
employer_name              0
municipality              53
region                    53
description_text           0
dtype: int64

In [14]:
df_clean["municipality"] = df_clean["municipality"].fillna("Unknown")
df_clean["region"] = df_clean["region"].fillna("Unknown")
df_clean.isnull().sum()

id                        0
headline                  0
publication_date          0
experience_required       0
occupation_label          0
occupation_group_label    0
occupation_field_label    0
employer_name             0
municipality              0
region                    0
description_text          0
dtype: int64